# Modelado Basico Bank Marketing

Notebook para entrenar un modelo base usando el flujo ya preprocesado. La validacion final sigue reservada para `Test`.

## Objetivo

Tomar el pipeline de preprocesamiento, entrenar un clasificador basal y comparar su desempeno con una linea base simple.

## Regla metodologica

1. Reutilizar el split y el preprocessing definidos en el notebook anterior.
2. Ajustar el pipeline solo con `Train`.
3. Evaluar primero un baseline simple.
4. Comparar con un clasificador supervisado.
5. Reportar metricas sobre `Test` sin tocarlo durante el ajuste.

In [6]:
from pathlib import Path
import sys

NOTEBOOK_CWD = Path.cwd().resolve()
for base in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]:
    for candidate in (base, base / "bank_marketing" / "notebooks"):
        if (candidate / "preprocessing.py").exists():
            sys.path.insert(0, str(candidate))
            break
    else:
        continue
    break
else:
    raise ModuleNotFoundError("No se encontro preprocessing.py")
import pandas as pd
import numpy as np
from IPython.display import display
import importlib
from sklearn.model_selection import train_test_split

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

import preprocessing as preprocessing_module
importlib.reload(preprocessing_module)
from preprocessing import load_data, split_data, build_preprocessing_pipeline

DATA_PATH = '../data/raw/bank-additional-full.csv'
TARGET = 'y'

df = load_data(DATA_PATH)
X_train, X_test, y_train, y_test = split_data(df, TARGET)

numeric_cols = ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
nominal_cols = ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
ordinal_cols = ['education']

preprocessor = build_preprocessing_pipeline(numeric_cols, nominal_cols, ordinal_cols)
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_p,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)

print('Shapes:')
print('X_fit:', X_fit.shape)
print('X_val:', X_val.shape)
print('X_test:', X_test_p.shape)


Shapes: (32950, 50) (8238, 50)


In [7]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, positive_label='yes'):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_te)
        pos_index = list(model.classes_).index(positive_label)
        score = proba[:, pos_index]
        roc = roc_auc_score((y_te == positive_label).astype(int), score)
    else:
        roc = np.nan
    return {
        'accuracy': accuracy_score(y_te, pred),
        'precision': precision_score(y_te, pred, pos_label=positive_label, zero_division=0),
        'recall': recall_score(y_te, pred, pos_label=positive_label, zero_division=0),
        'f1': f1_score(y_te, pred, pos_label=positive_label, zero_division=0),
        'roc_auc': roc,
        'confusion_matrix': confusion_matrix(y_te, pred),
        'report': classification_report(y_te, pred, zero_division=0),
    }

baseline = DummyClassifier(strategy='most_frequent')
baseline_result = evaluate_model(baseline, X_fit, y_fit, X_val, y_val)
display(pd.DataFrame([baseline_result]).drop(columns=['confusion_matrix', 'report']))
print(baseline_result['confusion_matrix'])
print(baseline_result['report'])


,accuracy,precision,recall,f1,roc_auc
0,0.887351,0.0,0.0,0.0,0.5


[[7310    0]
 [ 928    0]]
              precision    recall  f1-score   support

          no       0.89      1.00      0.94      7310
         yes       0.00      0.00      0.00       928

    accuracy                           0.89      8238
   macro avg       0.44      0.50      0.47      8238
weighted avg       0.79      0.89      0.83      8238



In [8]:
logit = LogisticRegression(max_iter=1000, class_weight='balanced')
logit_result = evaluate_model(logit, X_fit, y_fit, X_val, y_val)
display(pd.DataFrame([logit_result]).drop(columns=['confusion_matrix', 'report']))
print(logit_result['confusion_matrix'])
print(logit_result['report'])


,accuracy,precision,recall,f1,roc_auc
0,0.832969,0.363747,0.644397,0.465008,0.801189


[[6264 1046]
 [ 330  598]]
              precision    recall  f1-score   support

          no       0.95      0.86      0.90      7310
         yes       0.36      0.64      0.47       928

    accuracy                           0.83      8238
   macro avg       0.66      0.75      0.68      8238
weighted avg       0.88      0.83      0.85      8238



In [9]:
model_specs = {
    'baseline_majority': DummyClassifier(strategy='most_frequent'),
    'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'decision_tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
    'hist_gradient_boosting': HistGradientBoostingClassifier(random_state=42),
}

results = []
for name, model in model_specs.items():
    result = evaluate_model(model, X_fit, y_fit, X_val, y_val)
    result['model'] = name
    results.append(result)

results_df = pd.DataFrame(results).drop(columns=['confusion_matrix', 'report'])
results_df = results_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].sort_values('f1', ascending=False)
display(results_df)


,model,accuracy,precision,recall,f1,roc_auc
3,random_forest,0.867079,0.437265,0.627155,0.515272,0.812989
2,decision_tree,0.833576,0.366163,0.653017,0.469222,0.792639
1,logistic_regression,0.832969,0.363747,0.644397,0.465008,0.801189
4,hist_gradient_boosting,0.902646,0.678977,0.257543,0.373437,0.814258
0,baseline_majority,0.887351,0.000000,0.000000,0.000000,0.500000


In [10]:
from sklearn.metrics import precision_recall_curve

best_model_name = results_df.iloc[0]['model']
print('Mejor modelo según F1 en validación:', best_model_name)

best_model = model_specs[best_model_name]
best_model.fit(X_fit, y_fit)

if hasattr(best_model, 'predict_proba'):
    positive_label = 'yes'
    pos_index = list(best_model.classes_).index(positive_label)
    val_probs = best_model.predict_proba(X_val)[:, pos_index]
    precision, recall, thresholds = precision_recall_curve((y_val == positive_label).astype(int), val_probs)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

    threshold_results = pd.DataFrame({
        'threshold': thresholds,
        'precision': precision[:-1],
        'recall': recall[:-1],
        'f1': f1_scores[:-1],
    })
    display(threshold_results.sort_values('f1', ascending=False).head(5))

    print(f'Umbral con mayor F1 en validación: {best_threshold:.3f}')
    best_model.fit(X_train_p, y_train)
    test_probs = best_model.predict_proba(X_test_p)[:, pos_index]
    y_pred_thresh = np.where(test_probs >= best_threshold, positive_label, 'no')
    print('Métricas finales sobre Test con umbral validado:')
    print(classification_report(y_test, y_pred_thresh, zero_division=0))
    print('Matriz de confusión final:')
    print(confusion_matrix(y_test, y_pred_thresh))
else:
    print(f'El modelo {best_model_name} no soporta predict_proba, no se puede ajustar un umbral personalizado.')


Mejor modelo según F1 en validación: random_forest


,threshold,precision,recall,f1
6998,0.602720,0.483186,0.588362,0.530612
6997,0.601837,0.482759,0.588362,0.530355
7000,0.603004,0.483156,0.587284,0.530156
6996,0.600938,0.482332,0.588362,0.530097
6999,0.602759,0.482728,0.587284,0.529898


Umbral con mayor F1: 0.603
Métricas con umbral ajustado:
              precision    recall  f1-score   support

          no       0.95      0.92      0.93      7310
         yes       0.48      0.59      0.53       928

    accuracy                           0.88      8238
   macro avg       0.71      0.75      0.73      8238
weighted avg       0.89      0.88      0.89      8238

Matriz de confusión:
[[6726  584]
 [ 382  546]]


In [11]:
best_model_name = results_df.iloc[0]['model']
best_model_name


'random_forest'

## Lectura del modelo

El baseline sirve como piso metodológico. LogisticRegression actúa como primer clasificador interpretable para comparar el efecto del preprocesamiento y de las variables derivadas. La selección final se hace con validación interna para no usar `Test` como criterio de ajuste.


## Conclusión final

1. Se evaluaron baseline, regresión logística, árbol de decisión, bosque aleatorio e HistGradientBoosting sobre una partición interna derivada de `Train`.
2. El modelo con mejor F1 en validación fue `Random Forest`.
3. El umbral de decisión se ajustó con validación y luego se reportaron las métricas finales sobre `Test` una sola vez.
4. El resultado final es consistente con la defensa porque separa selección, ajuste y evaluación.
